In [1]:
from pydantic import BaseModel

from app.modules.product.models import Product
from sqlalchemy import *
from app.infra.database import AsyncSessionFactory

page = 1
page_size = 2
category = None

async with AsyncSessionFactory() as session:

    # 动态条件列表
    conditions = [Product.status == 'active']
    if category:
        conditions.append(Product.category == category)

    # 1.查询总条数 total
    # SELECT COUNT(id) FROM products
    # WHERE category = 'life' AND status = 'active'
    total = await session.scalar(
        select(func.count(Product.id))
        .where(*conditions)
    )
    print(total)


    # 2.查询当前页的数据

    # SELECT * FROM products
    # WHERE category = 'life' AND status = 'active'
    # LIMIT 2 OFFSET 0
    products = await session.scalars(
        select(Product) # SELECT * FROM products
        .where(*conditions)
        .offset((page - 1) * page_size).limit(page_size)
    )

    list = products.all()

    for p in list:
        print(p.id, p.name)


12
1 达尔文宝贝计划12号
2 达尔文12号重大疾病保险


In [28]:
print(list[0])

Product({'id': 1, 'category': 'critical_illness', 'image_url': 'https://files.huizecdn.com/file3/M00/91/12/rBYA3Gi1gZ2AXBL7AAYFJxRhbPg411.png', 'min_premium': Decimal('285.30'), 'target_group': '0-17周岁', 'status': 'active', 'updated_at': '2026-08-08 01:11:31', 'clause_name': '信美相互星辰守护B款少儿重疾险条款.pdf', 'name': '达尔文宝贝计划12号', 'insurer': '信美人寿相互保险社', 'description': '117种重疾+28种中症+45种轻症，少儿重疾保障灵活。', 'max_premium': Decimal('2501.83'), 'highlights': ['117种重疾', '少儿保障', '可选投保人豁免'], 'created_at': '2026-08-08 01:11:31'})


In [2]:
from app.modules.product.schemas import ProductResponse

# model_validate 是把一个dict转为 Pydantic的BaseModel
response = ProductResponse.model_validate(list[0])
print(response)

id=1 name='达尔文宝贝计划12号' clause_name='信美相互星辰守护B款少儿重疾险条款.pdf' category='critical_illness' insurer='信美人寿相互保险社' image_url='https://files.huizecdn.com/file3/M00/91/12/rBYA3Gi1gZ2AXBL7AAYFJxRhbPg411.png' description='117种重疾+28种中症+45种轻症，少儿重疾保障灵活。' min_premium=Decimal('285.30') max_premium=Decimal('2501.83') target_group='0-17周岁' highlights=['117种重疾', '少儿保障', '可选投保人豁免'] status='active'


In [3]:
print(type(response))

<class 'app.modules.product.schemas.ProductResponse'>
